# Modelado

## Importación de librerías y carga del dataset

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score

# Para guardar el modelo
import pickle

df = pd.read_csv('./data/titanic_procesado.csv')

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,1,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,0,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


## División de datos

Recordemos que estamos por implementar algoritmos de aprendizaje supervisado. Esto implica que necesitamos dividir nuestros datos en dos conjuntos: uno para entrenamiento y otro para pruebas. El conjunto de entrenamiento se utiliza para ajustar el modelo, mientras que el conjunto de pruebas se usa para evaluar su rendimiento y asegurarnos de que generaliza bien a datos no vistos. Esta división, conocida como división entrenamiento-prueba (train-test split).

El conjunto de entrenamiento se usará en el proceso de entrenamiento para que nuestro modelo pueda “aprenderse” las respuestas correctas a las predicciones que queremos realizar.

Utilizaremos la función train_test_split de Sckit Learn para realizar esta división.

Primero que nada, crearemos un DataFrame que contenga los datos sin la variable objetivo (target) Survived, y el otro contendrá únicamente esta variable objetivo Survived.

In [10]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [11]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

Hagamos la división entrenamiento-prueba.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train = X_train.values  # Convertir a NumPy array
y_train = y_train.values  # Convertir a NumPy array
X_test = X_test.values    # Convertir a NumPy array
y_test = y_test.values    # Convertir a NumPy array

# Imprimir dimensiones
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (712, 7)
y_train: (712,)
X_test: (179, 7)
y_test: (179,)


## Selección de modelo

Ésta es una de las grandes ventajas de implementar aprendizaje automático con herramientas modernas como Python y Scikit Learn. El código que implementaremos nos permitirá probrar todos estos modelos. Eligiremos el mejor modelo no a partir de nuestra intuición, sino con base en las métricas de precisión que resulte de cada uno en su fase de entrenamiento.

In [13]:
# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

Implementaremos una técnica llamada GridSearch, la cual nos permite encontrar los mejores hiperparámetros para nuestro modelo de aprendizaje automático. GridSearch explora exhaustivamente un conjunto predefinido de valores para cada hiperparámetro, entrenando y evaluando el modelo con cada combinación posible. Al final, selecciona la combinación que proporciona el mejor rendimiento según una métrica de evaluación específica, garantizando así que el modelo esté optimizado para obtener los mejores resultados posibles.

El primer paso para implementar GridSearch es crear un diccionario que contenga todos los modelos que queremos probar y los hiperparámetros que queramos probar en cada uno de estos. 

In [14]:
modelos = {
    'Logistic Regression': {
        'model': LogisticRegression(),
        'params': {
            'model__C': [0.1],
            'model__max_iter': [1000]
        }
    },
    'Support Vector Classifier': {
        'model': SVC(),
        'params': {
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Decision Tree Classifier': {
        'model': DecisionTreeClassifier(),
        'params': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Random Forest Classifier': {
        'model': RandomForestClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['auto', 'sqrt', 'log2']
        }
    },
    'Gradient Boosting Classifier': {
        'model': GradientBoostingClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'AdaBoost Classifier': {
        'model': AdaBoostClassifier(),
        'params': {
            'model__n_estimators': [10, 100]
        }
    },
    'K-Nearest Neighbors Classifier': {
        'model': KNeighborsClassifier(),
        'params': {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    'XGBoost Classifier': {
        'model': XGBClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3]
        }
    },
    'LGBM Classifier': {
        'model': LGBMClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    },
    'GaussianNB': {
        'model': GaussianNB(),
        'params': {}
    },
    'Naive Bayes Classifier': {
        'model': BernoulliNB(),
        'params': {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    }
}

## Entrenamiento

In [15]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'splitter': ['best', 'random'],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3],
            'learning_rate': [0.1, 0.2, 0.3],
            'verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'alpha': [0.1, 1.0, 10.0]
        }
    }
}

# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Inicializar GridSearchCV con el modelo y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)
    
    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)
    
    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)
    
    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_
    
    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

c:\Users\rope4001\OneDrive - NIQ\Documents\MLDSLearning\Hybridge\aprendizaje_supervisado_titanic\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
6      Clasificador K-Nearest Neighbors       0.83
7                  Clasificador XGBoost       0.82
8                     Clasificador LGBM       0.81
4     Clasificador de Gradient Boosting       0.81
3    Clasificador de Bosques Aleatorios       0.80
1   Clasificador de Vectores de Soporte       0.80
2     Clasificador de Árbol de Decisión       0.80
0                   Regresión Logística       0.79
5                 Clasificador AdaBoost       0.79
10             Clasificador Naive Bayes       0.79
9                            GaussianNB       0.76
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.83


c:\Users\rope4001\OneDrive - NIQ\Documents\MLDSLearning\Hybridge\aprendizaje_supervisado_titanic\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Inferencia

Terminamos nuestro proceso de entrenamiento. Lo que viene ahora es algo que llamamos “inferencia”. Esto significa que podremos usar nuestro modelo entrenado para predecir datos nuevos que no haya visto el modelo todavía. 

En el contexto de nuestro proyecto, podríamos ya usar nuestro modelo para alimentarle nueva información de pasajeros y predecir si sobrevivió o no. 

En nuestro proceso de entrenamiento, guardamos el mejor estimador en la variable `mejor_estimador`.

Para predecir nuevos valores, usaremos el método `predict` , mismo que recibe un numpy array como argumento, y éste debe ser de exactamente las mismas dimensiones que X_train y X_test.

Obtengamos los primeros datos de X_train y de y_train:

In [19]:
print(X_train[0])
print()
print(y_train[0])

[0.         1.         0.61581699 0.         0.         0.55559921
 1.        ]

0


Ahora creemos un nuevo numpy array con los datos que vemos en X_train[0]

In [20]:
nuevos_datos = np.array([0,1,0.6159084,0,0,0.55547282,1]).reshape(1,-1)

Y finalmente corramos predict

In [22]:
mejor_estimador.predict(nuevos_datos)

array([0])

## Guardar el modelo

Este último paso es necesario para la siguiente lección en la que pondremos nuestro modelo en producción. Usaremos un paquete de Python llamado Pickle, el cual nos permite serializar (guardar) objetos de Python en un archivo para luego poder cargarlos y utilizarlos en diferentes entornos, como una API o una aplicación web.

Pickle es particularmente útil cuando queremos guardar modelos entrenados o cualquier otro objeto complejo de Python. Al guardar el pipeline con Pickle, nos aseguramos de que todas las transformaciones de datos y el modelo en sí se conserven tal como fueron entrenados, permitiéndonos hacer predicciones consistentes con nuevos datos en producción sin necesidad de volver a aplicar las mismas transformaciones manualmente.

In [24]:
import pickle
import os

# Crear carpeta si no existe
os.makedirs("models", exist_ok=True)

# Guardar el mejor estimador en la carpeta models
with open("models/modelo.pkl", "wb") as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)
